In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Config

In [ ]:
REPO_ROOT = "/content/drive/MyDrive/Rutishauser_Lab/AI4Sci/METHODS/DiffusionBANMap/"
DIFFUSION_CONFIG_PATH = "config/diffusion_full_imagenet64_flow_uncond.yaml"   # relative to repo root
DIFFUSION_CKPT_PATH = None   # None → load best_ckpt.pt from experiment output dir
UNCONDITIONAL = True    # True → skip VAE / precomputed latents, pass z=0
LOAD_TEST_IMAGES = not UNCONDITIONAL                 
USE_EMA_WEIGHTS = True   # sample from the checkpoint's EMA shadow (models/ema.py) if
                         # present, falling back to the raw weights with a warning if the
                         # checkpoint predates ema_decay being set during training

N_SAMPLES = 8
GUIDANCE_SCALE = 1.0    # > 1.0 requires a CFG-trained model (cfg_uncond_prob > 0)
SEED = 42

# DDPM-only options (ignored when noise_process is 'flow' or 'vpsde')
SAMPLER = "ddpm"   # 'ddpm' or 'ddim'
ETA = 0.0          # ddim only: 0=deterministic, 1≈ddpm stochasticity

# Number of steps for ddim / flow / vpsde (ddpm always uses all T steps)
NUM_INFERENCE_STEPS = 100

import random, torch
random.seed(SEED)
torch.manual_seed(SEED)

## Imports

In [ ]:
import os
import sys
import types
import yaml
import random
import numpy as np
import subprocess
import matplotlib.pyplot as plt

sys.path.insert(0, REPO_ROOT)

import importlib.metadata
if not hasattr(importlib.metadata, '_patched'):
    _original_version = importlib.metadata.version
    def _patched_version(name):
        try:
            return _original_version(name)
        except importlib.metadata.PackageNotFoundError:
            return 'unknown'
    importlib.metadata.version = _patched_version
    importlib.metadata._patched = True

from models.beta_vae import BetaVAE
from models.unet import UNet
from models.noise_process import DDPM, RectifiedFlow, VPSDE
from utils import make_noise_schedule, diffusion_sample, flow_sample, vpsde_sample, make_run_tag

## Load models

In [ ]:
with open(os.path.join(REPO_ROOT, DIFFUSION_CONFIG_PATH)) as f:
    diff_cfg = types.SimpleNamespace(**yaml.safe_load(f))

diff_cfg.output_dir = os.path.join(REPO_ROOT, f"{diff_cfg.output_dir.lstrip('./')}_{make_run_tag(diff_cfg)}")
# betavae paths are absent in unconditional configs and in precomputed-latents configs
# (keys commented out in yaml)
if hasattr(diff_cfg, 'betavae_config_path'):
    diff_cfg.betavae_config_path = os.path.join(REPO_ROOT, diff_cfg.betavae_config_path.lstrip("./"))
if hasattr(diff_cfg, 'betavae_ckpt_path'):
    diff_cfg.betavae_ckpt_path   = os.path.join(REPO_ROOT, diff_cfg.betavae_ckpt_path.lstrip("./"))
if not os.path.isabs(diff_cfg.data_dir):
    diff_cfg.data_dir = os.path.join(REPO_ROOT, diff_cfg.data_dir.lstrip("./"))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# --- frozen β-VAE, only loaded when the diffusion model actually conditions on one ---
if UNCONDITIONAL:
    vae = None
    print("Unconditional mode: skipping β-VAE, z is a zero vector")
elif hasattr(diff_cfg, 'betavae_config_path'):
    with open(diff_cfg.betavae_config_path) as f:
        vae_cfg = types.SimpleNamespace(**yaml.safe_load(f))
    vae = BetaVAE(
        input_channels=vae_cfg.input_channels,
        input_height=vae_cfg.input_height,
        input_width=vae_cfg.input_width,
        latent_dim=vae_cfg.latent_dim,
        hidden_dim=vae_cfg.hidden_dim,
        beta=vae_cfg.beta,
        device=device,
    )
    vae_ckpt = torch.load(diff_cfg.betavae_ckpt_path, map_location=device)
    vae.load_state_dict(vae_ckpt["model_state_dict"])
    vae.eval()
    print(f"beta-VAE loaded (step {vae_ckpt['step']})")
else:
    # Conditional, but no VAE configured — the diffusion model was trained on precomputed
    # latents (e.g. AlexNet-fc6-PCA, see notebooks/alexnet_pca_latents.ipynb) row-aligned
    # with val_images.npy, so z is looked up from val_latents.npy instead of vae.encode()
    # (see "Load images" below).
    vae = None
    print("Conditional on precomputed latents — no VAE to load, z comes from val_latents.npy")

# --- UNet ---
unet = UNet(
    in_channels=diff_cfg.in_channels,
    image_size=diff_cfg.image_size,
    model_channels=diff_cfg.model_channels,
    channel_mult=diff_cfg.channel_mult,
    num_res_blocks=diff_cfg.num_res_blocks,
    attention_resolutions=diff_cfg.attention_resolutions,
    dropout=0.0,
    latent_dim=diff_cfg.latent_dim,
)

if DIFFUSION_CKPT_PATH is None:
    ckpt_dir = os.path.join(diff_cfg.output_dir, "checkpoints")
    DIFFUSION_CKPT_PATH = os.path.join(ckpt_dir, "best_ckpt.pt")
    if not os.path.exists(DIFFUSION_CKPT_PATH):
        raise FileNotFoundError(f"best_ckpt.pt not found in {ckpt_dir}")

diff_ckpt = torch.load(DIFFUSION_CKPT_PATH, map_location=device)
unet.load_state_dict(diff_ckpt["model_state_dict"])

if USE_EMA_WEIGHTS and "ema_state_dict" in diff_ckpt:
    from models.ema import ema_shadow_to_model_state_dict
    ema_sd = diff_ckpt["ema_state_dict"]
    unet.load_state_dict(ema_shadow_to_model_state_dict(unet, ema_sd["shadow"]))
    print(f"Sampling from EMA weights (EMA step {ema_sd['step']}, decay {ema_sd['decay']})")
elif USE_EMA_WEIGHTS:
    print("USE_EMA_WEIGHTS=True but checkpoint has no ema_state_dict "
          "(older run, or ema_decay wasn't set during training) — using raw weights")

unet = unet.to(device).eval()
print(f"UNet loaded: {DIFFUSION_CKPT_PATH}  (step {diff_ckpt['step']})")

## Load images

In [ ]:
if diff_cfg.dataset_type == "tiny_imagenet":
    subprocess.run(["wget", "http://cs231n.stanford.edu/tiny-imagenet-200.zip"], check=True)
    subprocess.run(["unzip", "-q", "tiny-imagenet-200.zip"], check=True)
    subprocess.run(["rm", "tiny-imagenet-200.zip"], check=True)
    subprocess.run(["ls", "/content/tiny-imagenet-200"], check=True)
elif diff_cfg.dataset_type == "imagenet64":
    subprocess.run(["mkdir", "-p", "/content/data/imagenet64/"])
    subprocess.run(["cp", os.path.join(REPO_ROOT, f"data/imagenet64/val_images.npy"), "/content/data/imagenet64/"], check=True)

    # Only stage precomputed latents (e.g. AlexNet-fc6-PCA) for runs that are actually
    # conditioned on them — same signal as the VAE-loading cell above, so an unconditional
    # or VAE-conditioned imagenet64 run never accidentally picks up (image, latent) pairs.
    if not diff_cfg.unconditional and not hasattr(diff_cfg, 'betavae_config_path'):
        subprocess.run([
            "cp", os.path.join(REPO_ROOT, f"data/imagenet64_alexnet_fc6_pca_latents/train_latents_z{diff_cfg.latent_dim}.npy"),
            diff_cfg.data_dir,
        ], check=True)
        subprocess.run([
            "cp", os.path.join(REPO_ROOT, f"data/imagenet64_alexnet_fc6_pca_latents/val_latents_z{diff_cfg.latent_dim}.npy"),
            diff_cfg.data_dir,
        ], check=True)
        print("Staged precomputed latents into data_dir — conditioning on precomputed latents")
    else:
        print("No precomputed-latents staging needed (unconditional or VAE-conditioned run)")

In [ ]:
if not UNCONDITIONAL and vae is not None and not LOAD_TEST_IMAGES:
    raise ValueError("β-VAE conditioning needs real images to encode a latent from — "
                      "set LOAD_TEST_IMAGES=True")

sampled_latents = None  # only populated in the precomputed-latents branch below
originals = None

if UNCONDITIONAL or vae is not None:
    if LOAD_TEST_IMAGES:
        # β-VAE conditioning (or an unconditional run that still wants images to display) —
        # encode raw images from a folder dataset (e.g. data/500Stimuli)
        import glob
        from torchvision import transforms
        from PIL import Image

        exts = {".jpg", ".jpeg", ".png", ".tif", ".tiff"}
        all_paths = sorted(
            p for p in glob.glob(os.path.join(diff_cfg.data_dir, "**", "*"), recursive=True)
            if os.path.splitext(p)[1].lower() in exts and "Zone.Identifier" not in p
        )
        sampled_paths = random.sample(all_paths, min(N_SAMPLES, len(all_paths)))

        transform = transforms.Compose([
            transforms.Resize((diff_cfg.image_size, diff_cfg.image_size)),
            transforms.ToTensor(),
        ])
        originals = torch.stack([transform(Image.open(p).convert("RGB")) for p in sampled_paths]).to(device)
    else:
        print("LOAD_TEST_IMAGES=False — skipping all image loading (unconditional sampling only)")
else:
    # Conditional on precomputed latents — z always comes from val_latents.npy (row-aligned
    # with val_images.npy, same convention as data_utils.py's _ImageNet64ArrayDataset), so
    # unlike β-VAE conditioning this never needs the real images. Set LOAD_TEST_IMAGES=False
    # to load *only* the latents and skip val_images.npy entirely (avoids the slow ImageNet64
    # image read on Colab); set it True to also pull the matching images for display / the
    # one-step-prediction diagnostic.

    val_latents_path = os.path.join(diff_cfg.data_dir, "val_latents.npy")
    val_latents = np.load(val_latents_path, mmap_mode='r')
    val_images_path = os.path.join(diff_cfg.data_dir, "val_images.npy")
    val_images = np.load(val_images_path, mmap_mode='r')
    print(f"Loaded precomputed latents + matching real images")

### Noise schedule

In [ ]:
np_type = getattr(diff_cfg, 'noise_process', 'ddpm')
embed_scale = getattr(diff_cfg, 'num_timesteps', 1000)

if np_type == 'ddpm':
    noise_proc = DDPM(
        num_timesteps=diff_cfg.num_timesteps,
        beta_schedule=diff_cfg.beta_schedule,
        min_snr_gamma=getattr(diff_cfg, 'min_snr_gamma', 5.0),
        device=device
    )
    schedule = make_noise_schedule(diff_cfg, device)
    print(f"DDPM  T={diff_cfg.num_timesteps}  beta_0={schedule['betas'][0]:.5f}  beta_T={schedule['betas'][-1]:.4f}")

elif np_type == 'flow':
    noise_proc = RectifiedFlow()
    schedule   = None
    print("RectifiedFlow — continuous ODE, t ∈ [0, 1]")

elif np_type == 'vpsde':
    noise_proc = VPSDE(
        beta_min=getattr(diff_cfg, 'vpsde_beta_min', 0.01),
        beta_max=getattr(diff_cfg, 'vpsde_beta_max', 5.0),
    )
    schedule   = None
    print(f"VPSDE  beta_min={noise_proc.beta_min}  beta_max={noise_proc.beta_max}")

else:
    raise ValueError(f"Unknown noise_process: {np_type!r}")

print(f"embed_scale={embed_scale}")

### Get conditioning latents (z)

In [ ]:
idx = random.sample(range(len(val_latents)), min(N_SAMPLES, len(val_latents)))
sampled_latents = torch.from_numpy(np.array(val_latents[idx])).float().to(device)

originals = torch.from_numpy(np.array(val_images[idx])).float().to(device) / 255.0
if diff_cfg.image_size != originals.shape[-1]:
    originals = torch.nn.functional.interpolate(
        originals, size=diff_cfg.image_size, mode='bilinear', align_corners=False
    )

In [ ]:
with torch.no_grad():
    if UNCONDITIONAL:
        n_z = originals.shape[0] if originals is not None else N_SAMPLES
        z = torch.zeros(n_z, diff_cfg.latent_dim, device=device)
        vae_recon = None
        print("Diffusion model is unconditional, skip beta VAE")
    elif vae is not None:
        z, _ = vae.encode(originals)
        vae_recon = torch.sigmoid(vae.decode(z)).cpu()
    else:
        # Conditional on precomputed latents — z was already looked up by index
        # alongside originals in "Load images", no VAE to reconstruct through.
        z = sampled_latents
        vae_recon = None
        print("Conditional on precomputed latents — no VAE reconstruction to show")

## One-step predictions

In [ ]:
if originals is None:
    print("Skipping one-step predictions — no test images loaded (set LOAD_TEST_IMAGES=True to enable)")
else:
    n_vis = originals.shape[0]
    t_vals = noise_proc.vis_t_vals()
    n_rows = 1 + 2 * len(t_vals)

    fig, axes = plt.subplots(n_rows, n_vis, figsize=(n_vis * 2, n_rows * 2), squeeze=False)

    def _show(ax, img):
        img = img.cpu().permute(1, 2, 0).float().clamp(0, 1).numpy()
        ax.imshow(img.squeeze(-1) if img.shape[-1] == 1 else img,
                  cmap='gray' if img.shape[-1] == 1 else None)
        ax.axis('off')

    row_labels = ['x0']
    for tv in t_vals:
        row_labels += [f'x_t  t={tv}', f'x_pred  t={tv}']

    for col in range(n_vis):
        _show(axes[0, col], originals[col])

    with torch.no_grad():
        for ti, t_val in enumerate(t_vals):
            t_tensor = torch.full((n_vis,), t_val, device=device, dtype=noise_proc.dtype)
            x_t, _ = noise_proc.corrupt(originals[:n_vis], t_tensor)
            pred = unet(x_t, noise_proc.embed_t(t_tensor, embed_scale), z[:n_vis])
            x0_pred = noise_proc.predict_x0(x_t, pred, t_tensor)
            row_xt = 1 + 2 * ti
            for col in range(n_vis):
                _show(axes[row_xt, col], x_t[col].clamp(0, 1))
                _show(axes[row_xt + 1, col], x0_pred[col].clamp(0, 1))

    for row, label in enumerate(row_labels):
        axes[row, 0].set_ylabel(label, fontsize=7, rotation=0, ha='right', va='center', labelpad=55)
    fig.suptitle(f'One-step predictions ({np_type})', fontsize=9)
    plt.tight_layout()
    plt.show()

### Sample images and generate

In [ ]:
with torch.no_grad():
    if np_type == 'ddpm':
        generated = diffusion_sample(
            z, unet, schedule, diff_cfg, device,
            sampler=SAMPLER, eta=ETA, num_inference_steps=NUM_INFERENCE_STEPS,
            guidance_scale=GUIDANCE_SCALE,
        )
    elif np_type == 'flow':
        generated = flow_sample(
            z, unet, diff_cfg, device,
            num_steps=NUM_INFERENCE_STEPS,
            guidance_scale=GUIDANCE_SCALE,
        )
    elif np_type == 'vpsde':
        generated = vpsde_sample(
            z, unet, diff_cfg, device,
            num_steps=NUM_INFERENCE_STEPS,
            guidance_scale=GUIDANCE_SCALE,
        )

generated = generated.cpu()
originals_cpu = originals.cpu() if originals is not None else None

In [ ]:
n = generated.shape[0]

if originals_cpu is None:
    rows = [generated]
    if UNCONDITIONAL:
        labels = ["diffusion (unconditional)"]
        title = "diffusion generated (unconditional) — no test images loaded"
    else:
        labels = ["diffusion (conditioned on precomputed latents)"]
        title = "diffusion generated — conditioned on precomputed latents, no test images loaded"
elif vae_recon is not None:
    rows = [originals_cpu, vae_recon, generated]
    labels = ["original", "β-VAE recon", "diffusion"]
    title = "original  |  β-VAE reconstruction  |  diffusion generated"
else:
    rows = [originals_cpu, generated]
    labels = ["original", "diffusion" if not UNCONDITIONAL else "diffusion (unconditional)"]
    title = "original  |  diffusion generated"

fig, axes = plt.subplots(len(rows), n, figsize=(n * 2.5, len(rows) * 2.5), squeeze=False)
fig.suptitle(title, fontsize=11)

for row_idx, (imgs, label) in enumerate(zip(rows, labels)):
    axes[row_idx, 0].set_ylabel(label, fontsize=9)
    for col_idx in range(n):
        axes[row_idx, col_idx].imshow(imgs[col_idx].permute(1, 2, 0).clamp(0, 1))
        axes[row_idx, col_idx].axis("off")

plt.tight_layout()
plt.show()

### Denoising progression (verbose)

In [ ]:
z_single = z[:1]

with torch.no_grad():
    if np_type == 'ddpm':
        _, snapshots = diffusion_sample(
            z_single, unet, schedule, diff_cfg, device,
            sampler=SAMPLER, eta=ETA, num_inference_steps=NUM_INFERENCE_STEPS,
            guidance_scale=GUIDANCE_SCALE,
            verbose=True, n_snapshots=10,
        )
    elif np_type == 'flow':
        _, snapshots = flow_sample(
            z_single, unet, diff_cfg, device,
            num_steps=NUM_INFERENCE_STEPS,
            guidance_scale=GUIDANCE_SCALE,
            verbose=True, n_snapshots=10,
        )
    elif np_type == 'vpsde':
        _, snapshots = vpsde_sample(
            z_single, unet, diff_cfg, device,
            num_steps=NUM_INFERENCE_STEPS,
            guidance_scale=GUIDANCE_SCALE,
            verbose=True, n_snapshots=10,
        )

n_snaps = len(snapshots)
fig, axes = plt.subplots(1, n_snaps, figsize=(n_snaps * 2, 2.8))

proc_label = SAMPLER.upper() if np_type == 'ddpm' else np_type.upper()
steps_label = (f"[{diff_cfg.num_timesteps} steps]"
               if np_type == 'ddpm' and SAMPLER == 'ddpm'
               else f"[{NUM_INFERENCE_STEPS} steps]")
fig.suptitle(f"{proc_label} progression  (left: noise → right: final)  {steps_label}", fontsize=11)

for ax, (t_idx, img) in zip(axes, snapshots):
    ax.imshow(img[0].permute(1, 2, 0).clamp(0, 1))
    ax.set_title(f"t={t_idx:.3f}" if isinstance(t_idx, float) else f"t={t_idx}", fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()